# Figures: station evaluation

Makes the station-level figures of the paper (Figs. 2, 4 and 5), which compare the hybrid model, the data-driven LSTM and ERA5-Land with the in situ NH-SWE observations at the held-out evaluation stations. The station metric tables are written instead by script: `src/3_analysis/compute_metrics.py` (Tables 3, 4 and 5).

**Before running:** create `src/common/local_paths.py` from `local_paths.example.py` (this notebook only uses `BASE_PATH`), and start Jupyter from the `notebooks/` folder.

**Inputs**, relative to `BASE_PATH`, and what produces them:

| File | Produced by |
|---|---|
| `data/STATION_METADATA.csv` | NH-SWE station metadata, with the `SnowClass` column from `src/1_preprocessing/extract_station_attributes.py` |
| `data/swe_data/NH_SWE/files/NH_SWE_ID_<id>.csv` | the NH-SWE dataset (observed SWE) |
| `data/swe_data/ERA5_LAND/NH_ERA5_SWE_ID_<id>.csv` | `src/1_preprocessing/extract_era5_swe_timeseries.py` |
| `data/valid_swe_regression_flags.nc` | `src/1_preprocessing/create_validity_mask.py` (`is_valid` is the study area) |
| `results/final_model_comparison/hybrid/station_splits.csv` | `src/2_modelling/main.py` (the split is the same for both models) |
| `results/final_model_comparison/{hybrid,lstm}/predictions/NH_pSWE_eval.csv` | `src/2_modelling/main.py`, the final run of each model |

The land polygons of Fig. 2 are downloaded from Natural Earth when the cell runs.

Each figure section only needs *Setup* and *Load data* to have run.

## Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# Paths: all files are read from the project folder BASE_PATH (data/ and results/),
# set per machine in src/common/local_paths.py (copy local_paths.example.py)
import sys
sys.path.append(str(Path.cwd().parent / 'src' / 'common'))
import local_paths

In [ ]:
# Define a style for the different data sources
style = {
    'Observed':  {'color': '#000000', 'linewidth': 3, 'label': 'Observed'},  # Black
    'LSTM':      {'color': '#2563EB', 'linewidth': 2, 'label': 'Data-driven'},  # Vivid Cobalt Blue
    'Hybrid':    {'color': '#DAA520', 'linewidth': 2, 'label': 'Hybrid'},  # Goldenrod Yellow
    'ERA5-Land': {'color': '#00C49F', 'linewidth': 2, 'label': 'ERA5-Land'}   # Bright Cyan-Green / Aquamarine
}

# Define the labels for the figure panels
panel_labels = [f"({letter})" for letter in "abcdefghijklmnopqrstuvwxyz"]

## Load data

In [ ]:
import pandas as pd

# Read the metadata
metadata = pd.read_csv(local_paths.BASE_PATH / 'data' / 'STATION_METADATA.csv', index_col=0)

# Check if there is any longitude above 180
if metadata['LON'].max() > 180:
    raise ValueError("There are longitude values above 180 in the metadata.")

# Check if there is any latitude below 0
if metadata['LAT'].min() < 0:
    raise ValueError("There are latitude values below 0 in the metadata.")

metadata.head()

In [ ]:
# Load the evaluation and train station IDs (the split is the same for all runs)
station_splits = pd.read_csv(local_paths.BASE_PATH / 'results' / 'final_model_comparison' / 'hybrid' / 'station_splits.csv', index_col=0)

# Gather the metadata of the evaluation stations and of all the others (train and early-stopping)
train_val_metadata = metadata.loc[station_splits[station_splits['split'] != 'eval'].index]
eval_metadata = metadata.loc[station_splits[station_splits['split'] == 'eval'].index]

eval_metadata.head()

In [ ]:
# Load the eval predictions for hybrid and LSTM models
hybrid_preds = pd.read_csv(local_paths.BASE_PATH / 'results' / 'final_model_comparison' / 'hybrid' / 'predictions' / f'NH_pSWE_eval.csv', parse_dates=[0])
lstm_preds = pd.read_csv(local_paths.BASE_PATH / 'results' / 'final_model_comparison' / 'lstm' / 'predictions' / f'NH_pSWE_eval.csv', parse_dates=[0])

# Check that the station IDs in the predictions match the eval metadata
pred_station_ids = set(hybrid_preds['station_id'].unique())
eval_station_ids = set(eval_metadata.index)
if not pred_station_ids == eval_station_ids:
    raise ValueError("Mismatch between prediction station IDs and eval metadata station IDs")

hybrid_preds.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the ERA5-Land SWE timeseries for the eval stations in parallel
era5_eval_timeseries = {}
dir_path = local_paths.BASE_PATH / 'data' / 'swe_data' / 'ERA5_LAND'
def _load_file(station_id):
    file_path = dir_path / f'NH_ERA5_SWE_ID_{station_id}.csv'
    df = pd.read_csv(file_path, parse_dates=[0])
    df.rename(columns={df.columns[0]: 'Date', 'SWE (mm)': 'pred_swe'}, inplace=True)
    df['station_id'] = station_id
    return station_id, df
max_workers = 16
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_load_file, station_id): station_id for station_id in eval_station_ids}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Loading ERA5-Land SWE timeseries for eval stations', unit='station'):
        station_id, df = fut.result()
        era5_eval_timeseries[station_id] = df

# Join the ERA5-Land SWE timeseries into a single DataFrame, in the same format as the predictions
era5_preds = pd.concat(era5_eval_timeseries, ignore_index=True)

era5_preds.head()

## Figure 2: study area and NH-SWE stations

The study area (`is_valid`) with the training and evaluation stations, in the Equal Earth projection.

In [ ]:
# Load the study area (is_valid in the validity flags) as the base map
inference_mask_ds = xr.open_dataset(local_paths.BASE_PATH / 'data' / 'valid_swe_regression_flags.nc')

In [ ]:
# Select the domain flag as a boolean array and count its pixels
inference_mask = inference_mask_ds['is_valid'].values.astype(bool)
print(f"Number of inference points: {np.sum(inference_mask)}")

# Lower the resolution of the base map for faster plotting (every 10th pixel)
downsample_factor = 10
plot_inference_mask = inference_mask[::downsample_factor, ::downsample_factor]
plot_lon_grid, plot_lat_grid = np.meshgrid(
    inference_mask_ds['lon'].values[::downsample_factor],
    inference_mask_ds['lat'].values[::downsample_factor],
)

# Count the training stations (every station not in the evaluation set)
num_train_stations = len(train_val_metadata)
num_total_stations = num_train_stations + len(eval_metadata)
pct_train_stations = (num_train_stations / num_total_stations) * 100
print(f"Number of training stations: {num_train_stations}/{num_total_stations} ({pct_train_stations:.1f}%)")

In [ ]:
from pyproj import Transformer
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path
import matplotlib.patheffects as pe


# =============================================================================
# Configuration
# =============================================================================
print("EPSG:8857 — Equal Earth")

plt.rcParams.update({"font.size": 12})

source_crs = "EPSG:4326"
target_crs = "EPSG:8857"

lon_min, lon_max = -180, 180
lat_min, lat_max = 0, 90

land_url = "https://naciscdn.org/naturalearth/110m/physical/ne_110m_land.zip"

gridline_color = "dimgray"
gridline_style = "--"
gridline_width = 0.4
gridline_alpha = 0.55

label_fontsize = 12
label_color = "dimgray"

boundary_color = "black"
boundary_width = 0.9


# =============================================================================
# Coordinate transformation
# =============================================================================
transformer = Transformer.from_crs(
    source_crs,
    target_crs,
    always_xy=True,
)

plot_x_grid, plot_y_grid = transformer.transform(
    plot_lon_grid,
    plot_lat_grid,
)


# =============================================================================
# Restrict gridded data to the displayed Northern Hemisphere domain
# =============================================================================
in_domain_grid = (
    (plot_lat_grid >= lat_min)
    & (plot_lat_grid <= lat_max)
)

inference_mask_plot = np.ma.masked_where(
    ~in_domain_grid,
    plot_inference_mask,
)


# =============================================================================
# Restrict and transform station coordinates
# =============================================================================
train_in_domain = (
    train_val_metadata.LAT.between(lat_min, lat_max)
    & train_val_metadata.LON.between(lon_min, lon_max)
)

eval_in_domain = (
    eval_metadata.LAT.between(lat_min, lat_max)
    & eval_metadata.LON.between(lon_min, lon_max)
)

train_x, train_y = transformer.transform(
    train_val_metadata.loc[train_in_domain, "LON"].to_numpy(),
    train_val_metadata.loc[train_in_domain, "LAT"].to_numpy(),
)

eval_x, eval_y = transformer.transform(
    eval_metadata.loc[eval_in_domain, "LON"].to_numpy(),
    eval_metadata.loc[eval_in_domain, "LAT"].to_numpy(),
)


# =============================================================================
# Read, clip, and transform land polygons
# =============================================================================
land_gdf = gpd.read_file(land_url)

if land_gdf.crs is None:
    land_gdf = land_gdf.set_crs(source_crs)

# Clip land to the domain in geographical coordinates before reprojection.
land_gdf = gpd.clip(land_gdf, (lon_min, lat_min, lon_max, lat_max)).to_crs(target_crs)

# =============================================================================
# Projected boundary of the displayed NH domain: 0°N–84°N
# =============================================================================
boundary_linewidth = 0.8
boundary_color = "black"

# Avoid exactly ±180° to prevent a possible antimeridian wrap artefact.
lon_left = -180 + 1e-6
lon_right = 180 - 1e-6

n_boundary = 1_000

# The four edges of the domain, which set the map extent and the border drawn below.
bottom_lon = np.linspace(lon_left, lon_right, n_boundary)
bottom_lat = np.full_like(bottom_lon, lat_min)

right_lat = np.linspace(lat_min, lat_max, n_boundary)
right_lon = np.full_like(right_lat, lon_right)

top_lon = np.linspace(lon_right, lon_left, n_boundary)
top_lat = np.full_like(top_lon, lat_max)

left_lat = np.linspace(lat_max, lat_min, n_boundary)
left_lon = np.full_like(left_lat, lon_left)

boundary_segments = [
    (bottom_lon, bottom_lat),
    (right_lon, right_lat),
    (top_lon, top_lat),
    (left_lon, left_lat),
]

# Transform all segments and retain their coordinates for the extent calculation.
projected_segments = []

for lon_segment, lat_segment in boundary_segments:
    segment_x, segment_y = transformer.transform(lon_segment, lat_segment)

    projected_segments.append((segment_x, segment_y))

# Calculate the map extent from the four projected boundary segments.
boundary_x = np.concatenate([segment[0] for segment in projected_segments])
boundary_y = np.concatenate([segment[1] for segment in projected_segments])

x_min, x_max = boundary_x.min(), boundary_x.max()
y_min, y_max = boundary_y.min(), boundary_y.max()

# =============================================================================
# Plot layers
# =============================================================================
fig, ax = plt.subplots(figsize=(18, 6), dpi=400)

inference_colormap = ListedColormap(["none", "darkgrey"])

land_gdf.plot(
    ax=ax,
    color="gainsboro",
    edgecolor="none",
    linewidth=0,
    zorder=0,
)

ax.pcolormesh(
    plot_x_grid,
    plot_y_grid,
    inference_mask_plot,
    cmap=inference_colormap,
    shading="auto",
    zorder=1,
)


# =============================================================================
# Longitude / latitude graticules
# =============================================================================
meridians = np.arange(-135, 180, 45)
parallels = np.arange(20, 85, 20)

lat_line = np.linspace(lat_min, lat_max, 400)
lon_line = np.linspace(lon_min, lon_max, 720)

# Meridians and their labels
for lon in meridians:
    grid_x, grid_y = transformer.transform(
        np.full_like(lat_line, lon),
        lat_line,
    )

    ax.plot(
        grid_x,
        grid_y,
        color=gridline_color,
        linewidth=gridline_width,
        linestyle=gridline_style,
        alpha=gridline_alpha,
        zorder=2,
    )

    if lon == 0:
        label = "0°"
    elif lon < 0:
        label = f"{abs(lon):.0f}°W"
    else:
        label = f"{lon:.0f}°E"

    # Label is placed outside the 0°N boundary without altering map extent.
    label_x, label_y = transformer.transform(lon, lat_min + 1.0)

    ax.annotate(
        label,
        xy=(label_x, label_y),
        xytext=(0, -10),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=label_fontsize,
        color=label_color,
        clip_on=False,
        zorder=6,
    )

# Parallels and their labels
for lat in parallels:
    grid_x, grid_y = transformer.transform(
        lon_line,
        np.full_like(lon_line, lat),
    )

    ax.plot(
        grid_x,
        grid_y,
        color=gridline_color,
        linewidth=gridline_width,
        linestyle=gridline_style,
        alpha=gridline_alpha,
        zorder=2,
    )

    # Label is placed outside the western boundary.
    label_x, label_y = transformer.transform(lon_min, lat)

    ax.annotate(
        f"{lat:.0f}°N",
        xy=(label_x, label_y),
        xytext=(-10, 0),
        textcoords="offset points",
        ha="right",
        va="center",
        fontsize=label_fontsize,
        color=label_color,
        clip_on=False,
        zorder=6,
    )


# =============================================================================
# Stations and projected-domain boundary
# =============================================================================
ax.scatter(
    train_x,
    train_y,
    s=18,
    marker="^",
    facecolor="cornflowerblue",
    edgecolor="none",
    zorder=3,
)

ax.scatter(
    eval_x,
    eval_y,
    s=25,
    marker="x",
    color="black",
    linewidth=0.5,
    alpha=0.7,
    zorder=4,
)

# Build one continuous, closed projected-domain boundary.
boundary_vertices = np.column_stack([boundary_x, boundary_y])

# Remove adjacent duplicate vertices at the four corners.
keep = np.r_[
    True,
    np.any(np.diff(boundary_vertices, axis=0) != 0, axis=1),
]
boundary_vertices = boundary_vertices[keep]

# Close the polygonal boundary.
boundary_vertices = np.vstack([
    boundary_vertices,
    boundary_vertices[0],
])

boundary_codes = np.full(
    len(boundary_vertices),
    Path.LINETO,
    dtype=np.uint8,
)
boundary_codes[0] = Path.MOVETO
boundary_codes[-1] = Path.CLOSEPOLY

boundary_path = Path(
    boundary_vertices,
    boundary_codes,
)

boundary_patch = PathPatch(
    boundary_path,
    facecolor="none",
    edgecolor="black",
    linewidth=1.0,
    joinstyle="round",
    capstyle="round",
    antialiased=True,
    zorder=6,
)

# White under-stroke isolates the border from graticules and map layers.
boundary_patch.set_path_effects([
    pe.Stroke(linewidth=2.0, foreground="white"),
    pe.Normal(),
])

ax.add_patch(boundary_patch)

# =============================================================================
# Legend and final layout
# =============================================================================
legend_elements = [
    Patch(facecolor="darkgray", edgecolor="dimgray", label="Study Area"),
    Line2D(
        [0], [0],
        marker="^",
        color="cornflowerblue",
        label="Train Stations (n={})".format(num_train_stations),
        markersize=10,
        linestyle="None",
    ),
    Line2D(
        [0], [0],
        marker="x",
        color="black",
        label="Evaluation Stations (n={})".format(len(eval_metadata)),
        markersize=10,
        linestyle="None",
    ),
]

ax.legend(
    handles=legend_elements,
    loc="lower left",
    bbox_to_anchor=(0.02, 0.02),
    fontsize=16,
    frameon=False,
)

# Strict projected extent: do not add geographic padding here.
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect("equal")

# Graticule labels replace standard projected-coordinate axes.
ax.set_axis_off()

# Space for exterior labels; avoid tight_layout() with exterior annotations.
fig.subplots_adjust(
    left=0.06,
    right=0.99,
    bottom=0.13,
    top=0.98,
)

plt.show()

## Figure 4: SWE time series at six evaluation stations

One random evaluation station per snow class, each shown over five snow years.

In [ ]:
# Choose one random station per snow class
stations_to_compare = []
snow_classes = []
for snow_class in eval_metadata['SnowClass'].unique():
    station_id = int(eval_metadata[eval_metadata['SnowClass'] == snow_class].sample(n=1, random_state=24).index[0])
    stations_to_compare.append(station_id)
    snow_classes.append(snow_class)

print(f"Comparing predictions for station IDs: {stations_to_compare}")

In [ ]:
if not all(station in eval_metadata.index for station in stations_to_compare):
    stations_not_found = [station for station in stations_to_compare if station not in eval_metadata.index]
    raise ValueError(f"One or more station IDs not found in eval metadata: {stations_not_found}")

plot_data_stations = {}

for station_to_compare in stations_to_compare:
    print(f"Comparing predictions for station ID: {station_to_compare}, Location: ({eval_metadata.loc[station_to_compare, 'LAT']:.4f}, {eval_metadata.loc[station_to_compare, 'LON']:.4f})")

    # Select the predictions for the station
    hybrid_station_preds = hybrid_preds[hybrid_preds['station_id'] == station_to_compare].set_index('Date')
    hybrid_station_preds.rename(columns={'pred_swe': 'Hybrid'}, inplace=True)
    lstm_station_preds = lstm_preds[lstm_preds['station_id'] == station_to_compare].set_index('Date')
    lstm_station_preds.rename(columns={'pred_swe': 'LSTM'}, inplace=True)
    era5_station_preds = era5_preds[era5_preds['station_id'] == station_to_compare].set_index('Date')
    era5_station_preds.rename(columns={'pred_swe': 'ERA5-Land'}, inplace=True)

    # Load the targets for the station
    targets_dir = local_paths.BASE_PATH / 'data' / 'swe_data' / 'NH_SWE' / 'files'
    target_file = targets_dir / f'NH_SWE_ID_{station_to_compare}.csv'
    target_station = pd.read_csv(target_file, parse_dates=[0], index_col=0)[['SWE (mm)']]
    target_station.rename(columns={'SWE (mm)': 'Observed'}, inplace=True)

    # Merge all data for plotting
    plot_data = target_station.join(era5_station_preds[['ERA5-Land']], how='outer')
    plot_data = plot_data.join(lstm_station_preds[['LSTM']], how='outer')
    plot_data = plot_data.join(hybrid_station_preds[['Hybrid']] , how='outer')

    plot_data_stations[station_to_compare] = plot_data

In [ ]:
# Five snow years shown for each station (from October of the year before the first one)
plot_years = {55457: [2006, 2010], 12252: [1981, 1985], 53218: [2011, 2015], 50541: [2001, 2005], 30487: [2014, 2018], 40455: [1990, 1994]}

In [ ]:
# Set the font size for the plot
plt.rcParams.update({'font.size': 16})

data_sources = style.keys()  # ['Observed', 'ERA5-Land', 'LSTM', 'Hybrid']

# Plot the different predictions and observations for the station
fig, ax = plt.subplots(len(stations_to_compare), 1, figsize=(21, 2 + len(stations_to_compare) * 3), dpi=400, gridspec_kw={
        'top': 0.95,    # Caps the top of the plot grid (leaves room for titles)
        'bottom': 0.1, # Leaves room for x-labels
        'hspace': 0.4,   # Explicitly forces vertical spacing between rows
        'wspace': 0.4    # Explicitly forces horizontal spacing between columns
    })
if len(stations_to_compare) == 1:
    ax = [ax]
for i, station in enumerate(stations_to_compare):

    plot_data = plot_data_stations[station][data_sources].copy()

    # Take the subset of the data for the specified years if provided
    if station in plot_years:
        start_year, end_year = plot_years[station]
        start_date = pd.to_datetime(f'{start_year-1}-10-01')
        end_date = pd.to_datetime(f'{end_year}-10-01')
        plot_data = plot_data.loc[start_date:end_date]

    snow_class = eval_metadata.loc[station, 'SnowClass']
    ax[i].set_title(f'Station ID {station}: {metadata.loc[station, "STANAME"].replace("_", " ")} ({snow_class})', fontsize=18)


    # Select the first and last available snow years for plotting
    def get_snow_year(date):
        return date.year if date.month >= 10 else date.year - 1
    first_year = get_snow_year(plot_data.dropna().index[0])
    last_year = get_snow_year(plot_data.dropna().index[-1])
    # print(f"Available data from {first_year} to {last_year}")

    if plot_years:
        if first_year > plot_years.get(station, [first_year, last_year])[0] or last_year < plot_years.get(station, [first_year, last_year])[1]:
            raise ValueError(f"Specified plot years {plot_years} are not within the available data range {first_year}-{last_year}.")
        first_year, last_year = plot_years.get(station, [first_year, last_year])
        # print(f"Plotting only snow years {first_year} to {last_year} as specified")

    for column in plot_data.columns:
        plot_column_data = plot_data[column].copy()
        if column != 'Observed':
            plot_column_data[plot_column_data < 5] = 0
        plot_column_data.plot(ax=ax[i], color=style[column]['color'], linewidth=style[column]['linewidth'], xlabel="", label=style[column]['label'])
    ax[i].set_ylim(bottom=0)
    ax[i].set_xlim(pd.to_datetime(f'{first_year-1}-10-01'), pd.to_datetime(f'{last_year}-10-01'))

# Add a subplot label (a, b, c, ...) to each subplot inside the subplot area
for i, ax_i in enumerate(ax):
    ax_i.text(0.01, 0.95, panel_labels[i], transform=ax_i.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')


ax[len(stations_to_compare)-1].set_xlabel('Date')
plt.text(0.075, 0.5, 'SWE (mm)', rotation=90, va='center', ha='center', transform=fig.transFigure)
ax[0].legend(loc='upper right', fontsize=16)
# plt.tight_layout()
plt.show()

## Figure 5: RMSE per evaluation station

Boxplots of the per-station RMSE, grouped by elevation band (a) and snow class (b).

In [ ]:
# Compute the RMSE of each model at each evaluation station, over the days with an observation
def rmse(observed, predicted):
    """Calculate Root Mean Square Error."""
    mask = ~np.isnan(observed) & ~np.isnan(predicted)
    observed = observed[mask]
    predicted = predicted[mask]
    if len(observed) == 0:
        raise RuntimeError('Cannot compute RMSE: no overlapping observed and predicted values')
    return np.sqrt(np.mean((observed - predicted) ** 2))

model_preds = {
    'ERA5-Land': era5_preds,
    'Hybrid': hybrid_preds,
    'LSTM': lstm_preds
}
station_metrics = {'rmse': {model: {} for model in model_preds.keys()}}

for station_id in tqdm(eval_metadata.index, desc='Calculating metrics for eval stations', unit='station'):

    # Load the target SWE data for the station
    targets = pd.read_csv(local_paths.BASE_PATH / 'data' / 'swe_data' / 'NH_SWE' / 'files' / f'NH_SWE_ID_{station_id}.csv', parse_dates=[0], index_col=[0])[['SWE (mm)']]

    for model_name, preds_df in model_preds.items():

        # Get the predictions for the station
        station_preds = preds_df[preds_df['station_id'] == station_id].set_index('Date')[['pred_swe']]
        data = station_preds.join(targets, how='outer').dropna()

        # If there are no overlapping data points, skip this station
        if data.empty:
            print(f"Warning: No overlapping data for station {station_id}, skipping metric calculation")
            continue

        station_metrics['rmse'][model_name][station_id] = rmse(data['SWE (mm)'].values, data['pred_swe'].values)

In [ ]:
import matplotlib.patches as mpatches

# Metric shown in the boxplots
metric_to_show = 'rmse'
metric_name = metric_to_show.upper()

# Plot the elevation and snow class boxplot in two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300, sharey=True,
                         gridspec_kw={'width_ratios': [1, 1.5]})

# Bands and models
elev_band_order = ['0-500', '500-1500', '1500+']
snow_class_order = ['Boreal Forest', 'Ephemeral', 'Maritime', 'Montane Forest', 'Prairie', 'Tundra']
models = ['LSTM', 'Hybrid', 'ERA5-Land']

# Prepare data for elevation bands
data_to_plot_elev = []
n_samples_elev = []
for i, band in enumerate(elev_band_order):
    for j, model in enumerate(models):
        n_samples_per_model = []
        model_dict = station_metrics.get(metric_to_show, {}).get(model, {})
        vals = [
            model_dict.get(station_id, np.nan)
            for station_id in eval_metadata.index
            if (
                (eval_metadata.loc[station_id, 'ELEV'] < 500 and band == '0-500')
                or (500 <= eval_metadata.loc[station_id, 'ELEV'] < 1500 and band == '500-1500')
                or (eval_metadata.loc[station_id, 'ELEV'] >= 1500 and band == '1500+')
            )
        ]
        # Drop NaNs for plotting; if empty, leave a single NaN so boxplot can render a placeholder
        vals = [v for v in vals if not (isinstance(v, float) and np.isnan(v))]
        if len(vals) == 0:
            vals = [np.nan]
        data_to_plot_elev.append(vals)
        n_samples_per_model.append(len(vals))

    # Check that all models have the same number of samples for this band
    if len(set(n_samples_per_model)) > 1:
        print(f"Warning: Different number of samples for models in elevation band {band}: {n_samples_per_model}. Taking the max for display.")
    n_samples_elev.append(np.max(n_samples_per_model))

# Compute group centers for x-ticks
centers_elev = [i * (len(models) + 1) + (len(models) + 1) / 2 for i in range(len(elev_band_order))]

# Prepare data for snow classes
data_to_plot_snow = []
n_samples_snow = []
for i, band in enumerate(snow_class_order):
    for j, model in enumerate(models):
        n_samples_per_model = []
        model_dict = station_metrics.get(metric_to_show, {}).get(model, {})
        vals = [
            model_dict.get(station_id, np.nan)
            for station_id in eval_metadata.index
            if (
                (eval_metadata.loc[station_id, 'SnowClass'] == band)
            )
        ]
        # Drop NaNs for plotting; if empty, leave a single NaN so boxplot can render a placeholder
        vals = [v for v in vals if not (isinstance(v, float) and np.isnan(v))]
        if len(vals) == 0:
            vals = [np.nan]
        data_to_plot_snow.append(vals)
        n_samples_per_model.append(len(vals))

    # Check that all models have the same number of samples for this band
    if len(set(n_samples_per_model)) > 1:
        print(f"Warning: Different number of samples for models in snow class {band}: {n_samples_per_model}. Taking the max for display.")
    n_samples_snow.append(np.max(n_samples_per_model))

# Compute group centers for x-ticks
centers_snow = [i * (len(models) + 1) + (len(models) + 1) / 2 for i in range(len(snow_class_order))]

# Set fontsizes
plt.rcParams.update({'font.size': 16})

# Plot elevation bands boxplot
bp_elev = axes[0].boxplot(data_to_plot_elev, positions=[i * (len(models) + 1) + j + 1 for i in range(len(elev_band_order)) for j in range(len(models))], widths=0.6, patch_artist=True, manage_ticks=False, showfliers=False)

# Color boxes by model (alternating)
for idx, box in enumerate(bp_elev['boxes']):
    model_name = models[idx % len(models)]
    box.set_facecolor(style[model_name]['color'])
    box.set_edgecolor('black')
for median in bp_elev['medians']:
    median.set_color('black')

# Set x-ticks and labels for elevation bands
axes[0].set_xticks(centers_elev)
axes[0].set_xticklabels([(band+f' (n={n_samples_elev[i]})').replace(' ', '\n') for i, band in enumerate(elev_band_order)])
axes[0].set_xlabel('Elevation Band', labelpad=20)
axes[0].set_ylabel(metric_name + ' (mm)')
axes[0].set_ylim(bottom=0, top=800)
# axes[0].set_ylim(top=1)

# Plot snow classes boxplot
bp_snow = axes[1].boxplot(data_to_plot_snow, positions=[i * (len(models) + 1) + j + 1 for i in range(len(snow_class_order)) for j in range(len(models))], widths=0.6, patch_artist=True, manage_ticks=False, showfliers=False)

# Color boxes by model (alternating)
for idx, box in enumerate(bp_snow['boxes']):
    model_name = models[idx % len(models)]
    box.set_facecolor(style[model_name]['color'])
    box.set_edgecolor('black')
for median in bp_snow['medians']:
    median.set_color('black')

# Set x-ticks and labels for snow classes
axes[1].set_xticks(centers_snow)
axes[1].set_xticklabels([(band+f' (n={n_samples_snow[i]})').replace(' ', '\n') for i, band in enumerate(snow_class_order)])
axes[1].set_xlabel('Snow Class')
# axes[1].set_ylabel(metric_name)
# axes[1].set_ylim(bottom=0)

# Legend
handles = [mpatches.Patch(facecolor=style[model]['color'], edgecolor='black', label=style[model]['label']) for model in models]
axes[1].legend(handles=handles, loc='upper right')

# Add subplot labels (a, b)
axes[0].text(0.075, 0.95, panel_labels[0], transform=axes[0].transAxes, fontweight='bold', va='top', ha='right')
axes[1].text(0.05, 0.95, panel_labels[1], transform=axes[1].transAxes, fontweight='bold', va='top', ha='right')

# Add horizontal gridlines behind the boxplots
for ax in axes:
    ax.set_axisbelow(True)  # draw grid behind boxes
    ax.grid(axis='y', which='major', linestyle='--', linewidth=1,
            color='gray', alpha=0.5)

plt.tight_layout()
plt.show()